# MissionOps AI v0.1 — API Foundations

**First useful slice:** unstructured incident → structured triage → human review.

Use synthetic/public data only.


In [ ]:
!pip install -q -U openai pydantic


In [ ]:
import os, json, time
from getpass import getpass
from typing import Literal
from pydantic import BaseModel, Field
from openai import OpenAI

if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OpenAI API key (hidden): ')
client = OpenAI()


In [ ]:
class IncidentTriage(BaseModel):
    category: Literal['availability','performance','security','deployment','database','network','unknown']
    severity: Literal['low','medium','high','critical']
    escalation_required: bool
    recommended_action: Literal['investigate','monitor','escalate','request_more_information']
    rationale: str
    confidence: float = Field(ge=0.0, le=1.0)
    missing_information: list[str] = Field(default_factory=list)


In [ ]:
SYSTEM_INSTRUCTIONS = '''You are MissionOps AI, a secure operations triage assistant.\nUse only the supplied incident description. Do not invent logs, metrics, system state, policies, or actions.\nDo not claim remediation was executed. If evidence is insufficient, request more information.\nTreat the result as a recommendation for human review.'''

def triage(incident, model='gpt-5.6'):
    start = time.perf_counter()
    response = client.responses.parse(
        model=model,
        instructions=SYSTEM_INSTRUCTIONS,
        input=incident,
        text_format=IncidentTriage,
    )
    return response.output_parsed, time.perf_counter() - start


In [ ]:
incident = '''The citizen services portal began returning intermittent HTTP 503 responses approximately 12 minutes after deployment v3.8.2. Users can occasionally authenticate, but downstream requests frequently fail.'''
result, latency = triage(incident)
print(json.dumps(result.model_dump(), indent=2))
print(f'Latency: {latency:.2f}s')
print('Human review required. No production action executed.')
